In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

C:\Users\Sahal\AppData\Local\Temp\ipykernel_7512\2611793992.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [ ]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [14]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemma4",
    model_provider="openai",
    api_key="dummy",
    base_url="http://localhost:8080/v1"
)

agent = create_agent(
    model=model,
    tools=[sql_query],
    system_prompt = """You are a SQLite database assistant with access to the `sql_query` tool.

        ENGINE & DIALECT:
        - Engine: SQLite.
        - Do NOT use MySQL syntax like `SHOW TABLES;`. Use `PRAGMA table_list;` or `PRAGMA table_info(table_name);`.

        CRITICAL CONSTRAINTS:
        1. NEVER GUESS FOREIGN KEYS OR COLUMNS:
        - Always inspect table columns before forming JOINs.
        - If two tables do not share a common key (e.g., `Artist` and `Track`), inspect intermediate tables (e.g., `Album`) to find connecting foreign keys.
        2. NO REPEATED QUERIES:
        - If a query returns an OperationalError or syntax error, NEVER submit the exact same query again.
        - Analyze the specific error, adjust syntax or schema references, and try an alternative approach.
        3. SQL CLAUSE ORDER ENFORCEMENT:
        - SQLite queries must strictly adhere to standard clause order:
            SELECT -> FROM -> [JOIN] -> WHERE -> GROUP BY -> HAVING -> ORDER BY -> LIMIT.

        WORKFLOW:
        1. Inspect schema: list tables and inspect column definitions of candidate tables.
        2. Trace the relationship path across foreign keys (inspect linking tables if needed).
        3. Execute the SELECT query with verified column names.
        4. Provide the answer based strictly on the retrieved data.
    """
)

In [16]:
from langchain.messages import HumanMessage

question = HumanMessage(content="List the first name, last name, and email of all customers from Brazil, sorted alphabetically by their last name?")

response = agent.invoke(
    {"messages": [question]}
)

In [6]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='4ef0d0d2-c33b-44a2-8313-0727993049ce'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 75, 'total_tokens': 113, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gemma-4-E4B-it-UD-Q4_K_XL.gguf', 'system_fingerprint': 'b9670-02810c7aa', 'id': 'chatcmpl-kH7z4HkELI5500mQUqbGs3UaowXL6TKo', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0856e-1bee-7f52-94b6-bc9f30ba7858-0', tool_calls=[{'name': 'sql_query', 'args': {'query': 'SELECT artist_name, COUNT(*) AS popularity FROM artists GROUP BY artist_name ORDER BY popularity DESC LIMIT 1;'}, 'id': 'FSF9KwldfbdELzee3pDTio3Hc9d0R9Eb', 'type': 'tool_call'}], invalid_tool_calls=[]

In [17]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT FirstName, LastName, Email FROM Customer WHERE Country = 'Brazil' ORDER BY LastName ASC;


In [ ]:
# Agentic Testing

import sqlite3
from typing import Any, Dict, List, Optional, Tuple


class TextToSQLEvalAgent:

    def __init__(self, db_path: str = "chinook.db"):
        self.db_path = db_path

    def _execute_query(
        self, query: str
    ) -> Tuple[bool, Optional[List[Tuple[Any, ...]]], Optional[str]]:
        """Executes a query in read-only mode and fetches normalized rows."""
        conn = None
        try:
            # Open via URI in read-only mode to prevent accidental writes/mutations
            conn = sqlite3.connect(
                f"file:{self.db_path}?mode=ro", uri=True, timeout=5.0
            )
            cursor = conn.cursor()
            cursor.execute(query)
            rows = cursor.fetchall()

            # Normalize floating points (e.g., 23.7600001 vs 23.76)
            normalized_rows = []
            for row in rows:
                norm_row = tuple(
                    round(item, 2) if isinstance(item, float) else item
                    for item in row
                )
                normalized_rows.append(norm_row)

            return True, normalized_rows, None
        except Exception as e:
            return False, None, str(e)
        finally:
            if conn:
                conn.close()

    def evaluate(
        self, prompt: str, generated_sql: str, ground_truth_sql: str
    ) -> Dict[str, Any]:
        """Compares the generated SQL output against ground truth execution results."""
        # 1. Clean query strings
        gen_clean = generated_sql.strip().rstrip(";")
        gt_clean = ground_truth_sql.strip().rstrip(";")

        # 2. Run Ground Truth Query
        gt_ok, gt_results, gt_err = self._execute_query(gt_clean)
        if not gt_ok:
            return {
                "prompt": prompt,
                "status": "ERROR",
                "verdict": "FAILED_GROUND_TRUTH",
                "error": f"Ground truth SQL failed: {gt_err}",
                "generated_sql": generated_sql,
            }

        # 3. Run Model's Generated Query
        gen_ok, gen_results, gen_err = self._execute_query(gen_clean)
        if not gen_ok:
            return {
                "prompt": prompt,
                "status": "SYNTAX_OR_EXEC_ERROR",
                "verdict": "FAIL",
                "error": gen_err,
                "generated_sql": generated_sql,
            }

        # 4. Compare Results (Order-agnostic comparison unless ordering was required)
        is_exact_match = gen_results == gt_results
        is_set_match = False
        try:
            is_set_match = set(gen_results) == set(gt_results)
        except TypeError:
            # In case unhashable types appear in results
            is_set_match = sorted(str(r) for r in gen_results) == sorted(
                str(r) for r in gt_results
            )

        passed = is_exact_match or is_set_match

        return {
            "prompt": prompt,
            "status": "SUCCESS",
            "verdict": "PASS" if passed else "FAIL",
            "is_exact_order_match": is_exact_match,
            "is_set_match": is_set_match,
            "gen_row_count": len(gen_results),
            "gt_row_count": len(gt_results),
            "gen_sample": gen_results[:3],
            "gt_sample": gt_results[:3],
            "generated_sql": generated_sql,
        }

    def run_suite(self, test_cases: List[Dict[str, str]]) -> Dict[str, Any]:
        """Evaluates a list of test dictionaries:

        [{'prompt': ..., 'generated_sql': ..., 'ground_truth_sql': ...}]
        """
        results = []
        passed_count = 0

        for tc in test_cases:
            res = self.evaluate(
                prompt=tc["prompt"],
                generated_sql=tc["generated_sql"],
                ground_truth_sql=tc["ground_truth_sql"],
            )
            if res["verdict"] == "PASS":
                passed_count += 1
            results.append(res)

        return {
            "total": len(test_cases),
            "passed": passed_count,
            "accuracy": (
                (passed_count / len(test_cases)) * 100 if test_cases else 0.0
            ),
            "details": results,
        }